ipwra estimators

In [ ]:
import openpyxl
import pandas as pd
import numpy as np
from io import StringIO
import datetime as dt
import econtools
import geopy.distance as geo
import pyreadstat
import pyarrow
import scipy
import pyreadr
import scipy.stats as stats
import statsmodels
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns
import os, shutil
from pathlib import Path
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

#use a list
packages = ['math','random','xlrd']
modules = map(__import__,packages)

from datetime import date
print("Today's date is:",date.today())

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from statsmodels.formula.api import glm
from sklearn.model_selection import train_test_split


# Load the data
# Step 1: Set the working directory (No direct equivalent, specify path)
data_path = "/Causal_Inference/Treatment_effects/data/cattaneo2.dta"
# Load the data (use pandas for this)
df = pd.read_stata(data_path,convert_categoricals=False)
# Print first few rows of the data
print(df.head())

# Step 2: Rename variables (using pandas)
df = df.rename(columns={'mbsmoke': 'treat'})

# Step 3: Fit the propensity score model (Logistic regression for the treatment variable)
X = df[['mmarried', 'mage', 'fbaby', 'medu']]  # Covariates
y = df['treat']  # Treatment variable

# Logistic regression model
logit_model = LogisticRegression()
logit_model.fit(X, y)

# Calculate propensity scores
df['pscore'] = logit_model.predict_proba(X)[:, 1]  # Get the probability of the treated class

# Step 4: Compute the inverse probability weights
df['ipw'] = np.where(df['treat'] == 0, 1 / (1 - df['pscore']), 1 / df['pscore'])

# Step 5: Fit the regression models for treated and control groups using inverse probability weights
# For treated group (treat == 1)
treated_data = df[df['treat'] == 1]
X_treat = treated_data[['mage', 'prenatal1', 'mmarried', 'fbaby']]
X_treat = sm.add_constant(X_treat)  # Add constant term (intercept)

model_treat = sm.WLS(treated_data['bweight'], X_treat, weights=treated_data['ipw']).fit()

# For control group (treat == 0)
control_data = df[df['treat'] == 0]
X_control = control_data[['mage', 'prenatal1', 'mmarried', 'fbaby']]
X_control = sm.add_constant(X_control)  # Add constant term (intercept)

model_control = sm.WLS(control_data['bweight'], X_control, weights=control_data['ipw']).fit()

# Print the regression results for both models
print(model_treat.summary())
print(model_control.summary())

# Step 6: Manually calculate the predictions for treated and control groups
# Using the model coefficients to predict outcomes
df['pom_t'] = (3201.664 + 
                 (-6.452 * df['mage']) + 
                 (26.590 * df['prenatal1']) + 
                 (136.652 * df['mmarried']) + 
                 (50.282 * df['fbaby']))

df['pom_c'] = (3187.79 + 
                 (3.15 * df['mage']) + 
                 (66.36 * df['prenatal1']) + 
                 (156.70 * df['mmarried']) + 
                 (-71.50 * df['fbaby']))

# Step 7: Summary statistics for treated and control group predictions
print("Summary of pom_c:")
print(df['pom_c'].describe())

print("Summary of pom_t:")
print(df['pom_t'].describe())

# Step 8: Calculate the treatment effect (difference in predictions)
df['ate'] = df['pom_t'] - df['pom_c']

# Calculate and print the mean of the 'ate' variable
mean_ate = df['ate'].mean()
print(f"Mean ATE: {mean_ate}")



In [ ]:
import datetime
print(f"ipwra estimator finished")
print(f"Program completed on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")